# 🚀 CODTECH Data Science Internship — All 4 Tasks

| Task | Topic |
|------|-------|
| Task 1 | Data Pipeline Development (ETL) |
| Task 2 | Deep Learning — Sentiment Analysis (LSTM) |
| Task 3 | End-to-End Project — Churn Prediction API |
| Task 4 | Optimization Model — Linear Programming |

> ✅ Run each cell one by one using **Shift + Enter**

---
# 📦 TASK 1: Data Pipeline Development (ETL)
**Tools:** Pandas, Scikit-learn  
**Goal:** Build an ETL pipeline — Extract, Transform, Load

In [ ]:
# ── Install dependencies ──
!pip install pandas numpy scikit-learn -q

In [ ]:
# ── TASK 1: IMPORTS ──
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

print('✅ Libraries imported successfully!')

In [ ]:
# ── STEP 1: EXTRACT — Generate Raw Dataset ──
np.random.seed(42)
n = 500

departments = ['Engineering', 'Marketing', 'HR', 'Finance', 'Sales', None]
genders     = ['Male', 'Female', 'male', 'FEMALE', None]

data = {
    'employee_id' : range(1, n+1),
    'age'         : np.random.randint(22, 60, n).astype(float),
    'gender'      : np.random.choice(genders, n),
    'department'  : np.random.choice(departments, n),
    'years_exp'   : np.round(np.random.uniform(0, 35, n), 2),
    'salary'      : np.round(np.random.normal(60000, 15000, n), 2),
    'performance' : np.random.choice([1, 2, 3, 4, 5, None], n),
    'satisfaction': np.round(np.random.uniform(1, 10, n), 2),
    'left_company': np.random.choice([0, 1], n),
}

df = pd.DataFrame(data)

# Introduce missing values
df.loc[np.random.choice(df.index, 30, replace=False), 'age']    = np.nan
df.loc[np.random.choice(df.index, 20, replace=False), 'salary'] = np.nan
# Introduce outliers
df.loc[np.random.choice(df.index, 5), 'salary'] = 500000

# Save raw dataset
df.to_csv('hr_raw_dataset.csv', index=False)

print(f'✅ Raw data shape: {df.shape}')
print(f'Missing values:\n{df.isnull().sum()[df.isnull().sum()>0]}')
df.head()

In [ ]:
# ── STEP 2a: CLEAN ──
df_clean = df.copy()
df_clean = df_clean.drop_duplicates()

# Standardise string casing
for col in ['gender', 'department']:
    df_clean[col] = df_clean[col].str.strip().str.title()

# Clip salary outliers using IQR
Q1, Q3  = df_clean['salary'].quantile(0.25), df_clean['salary'].quantile(0.75)
IQR     = Q3 - Q1
df_clean['salary'] = df_clean['salary'].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)

print(f'✅ After cleaning: {df_clean.shape}')
print(f'Remaining missing:\n{df_clean.isnull().sum()[df_clean.isnull().sum()>0]}')

In [ ]:
# ── STEP 2b: FEATURE ENGINEERING ──
df_eng = df_clean.copy()

df_eng['salary_per_year_exp'] = df_eng['salary'] / (df_eng['years_exp'] + 1)
df_eng['age_group']           = pd.cut(df_eng['age'], bins=[0,30,40,50,100],
                                        labels=['<30','30-40','40-50','50+'])
df_eng['high_performer']      = (df_eng['performance'] >= 4).astype(float)
df_eng['seniority']           = pd.cut(df_eng['years_exp'], bins=[-1,2,7,15,100],
                                        labels=['Junior','Mid','Senior','Principal'])

print('✅ New features created: salary_per_year_exp, age_group, high_performer, seniority')
df_eng[['salary_per_year_exp','age_group','high_performer','seniority']].head()

In [ ]:
# ── STEP 2c: SCIKIT-LEARN PIPELINE ──
TARGET = 'left_company'
X = df_eng.drop(columns=['employee_id', TARGET])
y = df_eng[TARGET]

numeric_cols     = X.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object','category']).columns.tolist()

num_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')),
                          ('scaler',  StandardScaler())])
cat_pipeline = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                          ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

preprocessor = ColumnTransformer([
    ('num', num_pipeline,   numeric_cols),
    ('cat', cat_pipeline,   categorical_cols)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                     random_state=42, stratify=y)
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

print(f'✅ Pipeline fitted!')
print(f'Train shape: {X_train_proc.shape} | Test shape: {X_test_proc.shape}')

In [ ]:
# ── STEP 3: LOAD — Save processed data ──
import os
os.makedirs('task1_output', exist_ok=True)

train_df = pd.DataFrame(X_train_proc)
train_df.insert(0, 'target', y_train.values)
train_df.to_csv('task1_output/train_processed.csv', index=False)

test_df = pd.DataFrame(X_test_proc)
test_df.insert(0, 'target', y_test.values)
test_df.to_csv('task1_output/test_processed.csv', index=False)

print('✅ TASK 1 COMPLETE!')
print(f'  Raw data shape      : {df.shape}')
print(f'  Processed train     : {X_train_proc.shape}')
print(f'  Processed test      : {X_test_proc.shape}')
print(f'  Files saved to      : task1_output/')

---
# 🧠 TASK 2: Deep Learning — Sentiment Analysis (LSTM)
**Tools:** TensorFlow/Keras  
**Goal:** Classify IMDB movie reviews as Positive or Negative

In [ ]:
!pip install tensorflow scikit-learn matplotlib -q

In [ ]:
# ── TASK 2: IMPORTS ──
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, ConfusionMatrixDisplay

np.random.seed(42)
tf.random.set_seed(42)

VOCAB_SIZE = 10000
MAX_LEN    = 200
EMBED_DIM  = 64
LSTM_UNITS = 64
BATCH_SIZE = 128
EPOCHS     = 10

print('✅ TensorFlow version:', tf.__version__)

In [ ]:
# ── STEP 1: LOAD IMDB DATASET ──
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

print(f'✅ Train: {len(X_train)} samples | Test: {len(X_test)} samples')
print(f'Positive reviews: {y_train.mean()*100:.1f}%')
print(f'Avg review length: {np.mean([len(r) for r in X_train]):.0f} words')

In [ ]:
# ── STEP 2: PREPROCESS — Pad sequences ──
X_train_pad = pad_sequences(X_train, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test,  maxlen=MAX_LEN, padding='post', truncating='post')

print(f'✅ Train padded: {X_train_pad.shape} | Test padded: {X_test_pad.shape}')

In [ ]:
# ── STEP 3: BUILD LSTM MODEL ──
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM, input_length=MAX_LEN),
    SpatialDropout1D(0.3),
    LSTM(LSTM_UNITS, dropout=0.2, recurrent_dropout=0.2),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# ── STEP 4: TRAIN ──
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    X_train_pad, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_test_pad, y_test),
    callbacks=[early_stop],
    verbose=1
)
print('✅ Training complete!')

In [ ]:
# ── STEP 5: EVALUATE ──
loss, acc = model.evaluate(X_test_pad, y_test, verbose=0)
print(f'✅ Test Accuracy : {acc*100:.2f}%')
print(f'✅ Test Loss     : {loss:.4f}')

y_pred_prob = model.predict(X_test_pad, verbose=0).flatten()
y_pred      = (y_pred_prob >= 0.5).astype(int)

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Negative','Positive']))

In [ ]:
# ── STEP 6: VISUALIZE RESULTS ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Task 2 — LSTM Sentiment Analysis Results', fontsize=15, fontweight='bold')

# Accuracy
axes[0,0].plot(history.history['accuracy'],     label='Train', color='steelblue', lw=2)
axes[0,0].plot(history.history['val_accuracy'], label='Val',   color='darkorange', lw=2, ls='--')
axes[0,0].set_title('Accuracy'); axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

# Loss
axes[0,1].plot(history.history['loss'],     label='Train', color='steelblue', lw=2)
axes[0,1].plot(history.history['val_loss'], label='Val',   color='darkorange', lw=2, ls='--')
axes[0,1].set_title('Loss'); axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

# Confusion Matrix
cm   = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Negative','Positive'])
disp.plot(ax=axes[1,0], colorbar=False, cmap='Blues')
axes[1,0].set_title('Confusion Matrix')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
roc_auc     = auc(fpr, tpr)
axes[1,1].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC={roc_auc:.3f}')
axes[1,1].plot([0,1],[0,1],'k--'); axes[1,1].set_title('ROC Curve')
axes[1,1].legend(); axes[1,1].grid(alpha=0.3)

plt.tight_layout()
os.makedirs('task2_output', exist_ok=True)
plt.savefig('task2_output/results.png', dpi=150)
plt.show()
print('✅ TASK 2 COMPLETE! Visualization saved.')

In [ ]:
# ── STEP 7: PREDICT ON CUSTOM REVIEWS ──
word_index = imdb.get_word_index()

def predict_review(text):
    tokens  = text.lower().split()
    encoded = [word_index.get(w, 2) + 3 for w in tokens]
    encoded = [i if i < VOCAB_SIZE else 2 for i in encoded]
    padded  = pad_sequences([encoded], maxlen=MAX_LEN, padding='post')
    prob    = model.predict(padded, verbose=0)[0][0]
    label   = '😊 POSITIVE' if prob >= 0.5 else '😞 NEGATIVE'
    conf    = prob if prob >= 0.5 else 1 - prob
    return label, conf

reviews = [
    'This movie was absolutely fantastic! Brilliant acting and captivating story.',
    'Terrible film. Complete waste of time. Boring and awful.',
    'One of the best movies I have ever seen. A true masterpiece.',
    'I fell asleep halfway through. Nothing interesting happens.',
]

print(f'\n{"Review":<55} {"Sentiment":<15} {"Confidence"}')
print('-'*80)
for r in reviews:
    label, conf = predict_review(r)
    print(f'{r[:52]:<55} {label:<15} {conf*100:.1f}%')

---
# 🌐 TASK 3: End-to-End Project — Churn Prediction API
**Tools:** Scikit-learn, FastAPI, Uvicorn  
**Goal:** Train a churn model and deploy it as a REST API

In [ ]:
!pip install fastapi uvicorn nest_asyncio requests scikit-learn pandas -q

In [ ]:
# ── TASK 3: IMPORTS ──
import pandas as pd
import numpy as np
import pickle, os, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
import matplotlib.pyplot as plt

os.makedirs('task3_output', exist_ok=True)
print('✅ Libraries ready!')

In [ ]:
# ── STEP 1: GENERATE DATASET ──
np.random.seed(42)
n = 1000

data = {
    'customer_id'      : [f'CUST{i:04d}' for i in range(1,n+1)],
    'tenure_months'    : np.random.randint(1, 72, n),
    'monthly_charges'  : np.round(np.random.uniform(20, 120, n), 2),
    'total_charges'    : np.round(np.random.uniform(100, 8000, n), 2),
    'num_products'     : np.random.randint(1, 6, n),
    'contract_type'    : np.random.choice(['Month-to-month','One year','Two year'], n),
    'payment_method'   : np.random.choice(['Credit card','Bank transfer','Electronic check','Mailed check'], n),
    'internet_service' : np.random.choice(['DSL','Fiber optic','No'], n),
    'tech_support'     : np.random.choice(['Yes','No'], n),
    'online_security'  : np.random.choice(['Yes','No'], n),
    'senior_citizen'   : np.random.choice([0,1], n, p=[0.84,0.16]),
    'dependents'       : np.random.choice(['Yes','No'], n),
    'paperless_billing': np.random.choice(['Yes','No'], n),
}

df3 = pd.DataFrame(data)

churn_prob = (
    0.3
    + (df3['contract_type']=='Month-to-month').astype(float)*0.25
    + (df3['tenure_months']<12).astype(float)*0.15
    + (df3['monthly_charges']>80).astype(float)*0.10
    - (df3['tech_support']=='Yes').astype(float)*0.10
    - (df3['online_security']=='Yes').astype(float)*0.08
).clip(0.05, 0.95)

df3['churn'] = (np.random.uniform(0,1,n) < churn_prob).astype(int)
df3.loc[np.random.choice(df3.index,30,replace=False),'total_charges']   = np.nan
df3.loc[np.random.choice(df3.index,15,replace=False),'monthly_charges'] = np.nan

df3.to_csv('task3_output/churn_raw_dataset.csv', index=False)
print(f'✅ Dataset: {df3.shape} | Churn rate: {df3.churn.mean()*100:.1f}%')
df3.head()

In [ ]:
# ── STEP 2: PREPROCESSING ──
df3_model = df3.drop(columns=['customer_id'])
X3 = df3_model.drop(columns=['churn'])
y3 = df3_model['churn']

numeric_cols3     = X3.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_cols3 = X3.select_dtypes(include=['object']).columns.tolist()

num_pipe3 = Pipeline([('imp', SimpleImputer(strategy='median')),   ('scl', StandardScaler())])
cat_pipe3 = Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                       ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

preprocessor3 = ColumnTransformer([
    ('num', num_pipe3, numeric_cols3),
    ('cat', cat_pipe3, categorical_cols3)
])

X3_train, X3_test, y3_train, y3_test = train_test_split(X3, y3, test_size=0.2,
                                                          random_state=42, stratify=y3)
X3_train_proc = preprocessor3.fit_transform(X3_train)
X3_test_proc  = preprocessor3.transform(X3_test)

print(f'✅ Train: {X3_train_proc.shape} | Test: {X3_test_proc.shape}')

In [ ]:
# ── STEP 3: TRAIN & SELECT BEST MODEL ──
models3 = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting'  : GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results3   = {}
best_model3, best_auc3, best_name3 = None, 0, ''

for name, m in models3.items():
    m.fit(X3_train_proc, y3_train)
    yp   = m.predict(X3_test_proc)
    ypr  = m.predict_proba(X3_test_proc)[:,1]
    acc  = accuracy_score(y3_test, yp)
    rauc = roc_auc_score(y3_test, ypr)
    results3[name] = {'acc': acc, 'auc': rauc, 'model': m}
    print(f'  {name:<25} Acc={acc*100:.2f}%  AUC={rauc:.4f}')
    if rauc > best_auc3:
        best_auc3, best_model3, best_name3 = rauc, m, name

print(f'\n✅ Best: {best_name3} (AUC={best_auc3:.4f})')

with open('task3_output/model.pkl','wb') as f:        pickle.dump(best_model3, f)
with open('task3_output/preprocessor.pkl','wb') as f: pickle.dump((preprocessor3, numeric_cols3, categorical_cols3), f)
print('✅ Model and preprocessor saved!')

In [ ]:
# ── STEP 4: EVALUATE & VISUALIZE ──
y3_pred = best_model3.predict(X3_test_proc)
y3_prob = best_model3.predict_proba(X3_test_proc)[:,1]

print(f'Classification Report ({best_name3}):')
print(classification_report(y3_test, y3_pred, target_names=['No Churn','Churn']))

fig, axes = plt.subplots(1,3,figsize=(16,5))
fig.suptitle(f'Task 3 — Churn Prediction ({best_name3})', fontsize=14, fontweight='bold')

# Model comparison
names3 = list(results3.keys())
aucs3  = [results3[n]['auc'] for n in names3]
bars   = axes[0].bar(names3, aucs3, color=['steelblue','darkorange','green'])
axes[0].set_title('Model Comparison (AUC)'); axes[0].set_ylim(0.5,1.0)
for b, v in zip(bars, aucs3): axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+0.005, f'{v:.3f}', ha='center', fontweight='bold')
axes[0].tick_params(axis='x', rotation=15)

# Confusion Matrix
cm3   = confusion_matrix(y3_test, y3_pred)
disp3 = ConfusionMatrixDisplay(cm3, display_labels=['No Churn','Churn'])
disp3.plot(ax=axes[1], colorbar=False, cmap='Blues'); axes[1].set_title('Confusion Matrix')

# ROC
fpr3, tpr3, _ = roc_curve(y3_test, y3_prob)
auc3          = auc(fpr3, tpr3)
axes[2].plot(fpr3, tpr3, color='steelblue', lw=2, label=f'AUC={auc3:.3f}')
axes[2].plot([0,1],[0,1],'k--'); axes[2].set_title('ROC Curve')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('task3_output/results.png', dpi=150)
plt.show()

In [ ]:
# ── STEP 5: DEPLOY AS FASTAPI + TEST IN NOTEBOOK ──
from fastapi import FastAPI
from pydantic import BaseModel
from typing import Literal
import nest_asyncio, uvicorn, threading, time, requests

nest_asyncio.apply()

# Load saved artifacts
with open('task3_output/model.pkl','rb') as f:        _model = pickle.load(f)
with open('task3_output/preprocessor.pkl','rb') as f: _prep, _num, _cat = pickle.load(f)

app = FastAPI(title='Churn Prediction API')

class CustomerData(BaseModel):
    tenure_months    : int
    monthly_charges  : float
    total_charges    : float
    num_products     : int
    contract_type    : Literal['Month-to-month','One year','Two year']
    payment_method   : Literal['Credit card','Bank transfer','Electronic check','Mailed check']
    internet_service : Literal['DSL','Fiber optic','No']
    tech_support     : Literal['Yes','No']
    online_security  : Literal['Yes','No']
    senior_citizen   : Literal[0,1]
    dependents       : Literal['Yes','No']
    paperless_billing: Literal['Yes','No']

@app.get('/')
def home(): return {'message': 'Churn Prediction API is running!'}

@app.post('/predict')
def predict(data: CustomerData):
    df_in  = pd.DataFrame([data.dict()])
    proc   = _prep.transform(df_in)
    pred   = int(_model.predict(proc)[0])
    prob   = float(_model.predict_proba(proc)[0][1])
    risk   = 'HIGH' if prob>=0.7 else 'MEDIUM' if prob>=0.4 else 'LOW'
    return {
        'churn_prediction' : pred,
        'churn_label'      : 'Will Churn' if pred==1 else 'Will NOT Churn',
        'churn_probability': round(prob,4),
        'risk_level'       : risk
    }

# Start server in background thread
def run(): uvicorn.run(app, host='0.0.0.0', port=8000, log_level='error')
t = threading.Thread(target=run, daemon=True)
t.start()
time.sleep(4)
print('✅ API Server started!')

In [ ]:
# ── TEST THE API ──
response = requests.post('http://localhost:8000/predict', json={
    'tenure_months':5, 'monthly_charges':95.5, 'total_charges':450.0,
    'num_products':2, 'contract_type':'Month-to-month',
    'payment_method':'Electronic check', 'internet_service':'Fiber optic',
    'tech_support':'No', 'online_security':'No',
    'senior_citizen':1, 'dependents':'No', 'paperless_billing':'Yes'
})

result = response.json()
print('✅ TASK 3 API RESPONSE:')
for k, v in result.items():
    print(f'  {k:<22}: {v}')

---
# 📊 TASK 4: Optimization Model — Linear Programming
**Tools:** PuLP  
**Goal:** Maximize manufacturing profit given machine and material constraints

In [ ]:
!pip install pulp -q

In [ ]:
# ── TASK 4: IMPORTS ──
import pulp
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
os.makedirs('task4_output', exist_ok=True)
print('✅ PuLP version:', pulp.__version__)

In [ ]:
# ── STEP 1: PROBLEM DEFINITION ──
print('''
BUSINESS PROBLEM:
  A company makes 4 products (A, B, C, D).
  Goal: Maximize weekly profit given machine hours and raw material limits.

CONSTRAINTS:
  • Machine 1: 240 hrs/week
  • Machine 2: 180 hrs/week
  • Machine 3: 150 hrs/week
  • Raw material: 400 kg/week
  • Min/Max demand per product
''')

products = ['Product_A', 'Product_B', 'Product_C', 'Product_D']

profit_per_unit = {'Product_A':25, 'Product_B':30, 'Product_C':15, 'Product_D':40}

machine_hours   = {
    'Product_A':[2.0,1.5,0.5],
    'Product_B':[1.0,2.0,1.0],
    'Product_C':[3.0,0.5,0.5],
    'Product_D':[1.5,2.5,2.0]
}

raw_material    = {'Product_A':3,'Product_B':4,'Product_C':2,'Product_D':5}
machine_cap     = [240,180,150]
material_limit  = 400
min_prod        = {'Product_A':10,'Product_B':10,'Product_C':5, 'Product_D':5}
max_prod        = {'Product_A':60,'Product_B':50,'Product_C':70,'Product_D':40}

# Display product info
df_products = pd.DataFrame({
    'Profit/Unit'    : profit_per_unit,
    'Machine1 hrs'   : {p:machine_hours[p][0] for p in products},
    'Machine2 hrs'   : {p:machine_hours[p][1] for p in products},
    'Machine3 hrs'   : {p:machine_hours[p][2] for p in products},
    'Material (kg)'  : raw_material,
    'Min Units'      : min_prod,
    'Max Units'      : max_prod,
})
df_products

In [ ]:
# ── STEP 2: BUILD & SOLVE LP MODEL ──
prob = pulp.LpProblem('Manufacturing_Profit_Maximization', pulp.LpMaximize)

# Decision variables
x = {p: pulp.LpVariable(f'units_{p}', lowBound=min_prod[p],
                          upBound=max_prod[p], cat='Continuous') for p in products}

# Objective: Maximize profit
prob += pulp.lpSum(profit_per_unit[p] * x[p] for p in products), 'Total_Profit'

# Machine constraints
for m in range(3):
    prob += (pulp.lpSum(machine_hours[p][m] * x[p] for p in products)
             <= machine_cap[m], f'Machine_{m+1}')

# Material constraint
prob += (pulp.lpSum(raw_material[p] * x[p] for p in products)
         <= material_limit, 'Raw_Material')

# Solve
prob.solve(pulp.PULP_CBC_CMD(msg=0))
print(f'✅ Status: {pulp.LpStatus[prob.status]}')

In [ ]:
# ── STEP 3: DISPLAY RESULTS ──
opt_prod     = {p: pulp.value(x[p]) for p in products}
total_profit = pulp.value(prob.objective)

print(f'\n{"Product":<15} {"Units":>10} {"Profit":>15}')
print('-'*42)
for p in products:
    u = opt_prod[p]
    print(f'{p:<15} {u:>10.1f} {u*profit_per_unit[p]:>14.0f}')
print('-'*42)
print(f'{"TOTAL PROFIT":<15} {"".rjust(10)} {total_profit:>14,.0f}')

print(f'\nRESOURCE UTILIZATION:')
print(f'{"Resource":<18} {"Used":>8} {"Capacity":>10} {"Util%":>8}')
print('-'*46)
for m in range(3):
    used = sum(machine_hours[p][m]*opt_prod[p] for p in products)
    util = used/machine_cap[m]*100
    print(f'{"Machine "+str(m+1):<18} {used:>8.1f} {machine_cap[m]:>10} {util:>7.1f}%')
mat_used = sum(raw_material[p]*opt_prod[p] for p in products)
print(f'{"Raw Material":<18} {mat_used:>8.1f} {material_limit:>10} {mat_used/material_limit*100:>7.1f}%')

In [ ]:
# ── STEP 4: SENSITIVITY ANALYSIS ──
d_profits, total_profits = [], []

for dp in range(20, 70, 5):
    mod_profit = {**profit_per_unit, 'Product_D': dp}
    ps = pulp.LpProblem('s', pulp.LpMaximize)
    xs = {p: pulp.LpVariable(f's_{p}', lowBound=min_prod[p], upBound=max_prod[p]) for p in products}
    ps += pulp.lpSum(mod_profit[p]*xs[p] for p in products)
    for m in range(3): ps += pulp.lpSum(machine_hours[p][m]*xs[p] for p in products) <= machine_cap[m]
    ps += pulp.lpSum(raw_material[p]*xs[p] for p in products) <= material_limit
    ps.solve(pulp.PULP_CBC_CMD(msg=0))
    d_profits.append(dp)
    total_profits.append(pulp.value(ps.objective))

print('Sensitivity Analysis — Product_D profit vs Total Profit:')
for dp, tp in zip(d_profits, total_profits):
    print(f'  Product_D = ₹{dp}/unit  →  Total = ₹{tp:,.0f}')

In [ ]:
# ── STEP 5: VISUALIZATIONS ──
fig, axes = plt.subplots(2, 2, figsize=(14,10))
fig.suptitle('Task 4 — Manufacturing Profit Optimization', fontsize=15, fontweight='bold')

# Production quantities
vals = [opt_prod[p] for p in products]
bars = axes[0,0].bar(products, vals, color=['steelblue','darkorange','green','crimson'])
axes[0,0].set_title('Optimal Production Quantities'); axes[0,0].set_ylabel('Units/Week')
for b,v in zip(bars,vals): axes[0,0].text(b.get_x()+b.get_width()/2, b.get_height()+0.3, f'{v:.1f}', ha='center', fontweight='bold')
axes[0,0].grid(axis='y', alpha=0.3)

# Profit contribution pie
contribs = [opt_prod[p]*profit_per_unit[p] for p in products]
axes[0,1].pie(contribs, labels=products, autopct='%1.1f%%',
              colors=['steelblue','darkorange','green','crimson'], startangle=90)
axes[0,1].set_title('Profit Contribution by Product')

# Resource utilization
resources = ['Machine 1','Machine 2','Machine 3','Raw Material']
util_vals = [
    sum(machine_hours[p][m]*opt_prod[p] for p in products)/machine_cap[m]*100 for m in range(3)
] + [sum(raw_material[p]*opt_prod[p] for p in products)/material_limit*100]
colors4 = ['green' if u<80 else 'orange' if u<95 else 'red' for u in util_vals]
axes[1,0].barh(resources, util_vals, color=colors4)
axes[1,0].axvline(100, color='red', ls='--', label='100% capacity')
axes[1,0].set_title('Resource Utilization (%)'); axes[1,0].legend(); axes[1,0].grid(axis='x',alpha=0.3)

# Sensitivity
axes[1,1].plot(d_profits, total_profits, marker='o', color='steelblue', lw=2)
axes[1,1].set_title('Sensitivity: Product_D Profit vs Total Profit')
axes[1,1].set_xlabel('Product_D Profit/Unit (₹)'); axes[1,1].set_ylabel('Total Profit (₹)')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('task4_output/optimization_results.png', dpi=150)
plt.show()
print('✅ TASK 4 COMPLETE!')

In [ ]:
# ── BUSINESS INSIGHTS ──
best_p = max(products, key=lambda p: opt_prod[p]*profit_per_unit[p])
print(f'''
✅ BUSINESS INSIGHTS:
  Maximum weekly profit         : ₹{total_profit:,.0f}
  Top revenue contributor       : {best_p}
  Top product contribution      : ₹{opt_prod[best_p]*profit_per_unit[best_p]:,.0f}

  Recommendations:
  • Prioritize {best_p} — highest profit per unit (₹{profit_per_unit[best_p]})
  • Negotiate higher demand cap for Product_D
  • Increasing raw material supply unlocks more profit
  • Re-run this model whenever prices or capacity changes
''')

---
# ✅ ALL 4 TASKS COMPLETE!

| Task | Status | Output |
|------|--------|--------|
| Task 1 — ETL Pipeline | ✅ Done | `task1_output/` |
| Task 2 — Deep Learning | ✅ Done | `task2_output/` |
| Task 3 — Churn API | ✅ Done | `task3_output/` |
| Task 4 — Optimization | ✅ Done | `task4_output/` |

**Organization:** CODTECH IT Solutions